# aiswakepy — Run Pipeline

Edit the paths and parameters in **Cell 1** before running.  
Each cell is independent — re-run any stage without restarting the kernel.

In [ ]:
# === EDIT THESE PATHS FOR YOUR RUN ===
ais_csv          = r"examples/ais/DT.csv"
coastline_shp    = r"examples/coastline/proposed_WC_dissolved_v2_WGS84.shp"
bathy_file       = r"examples/bathymetry/61803960_WestCoast_HD_25m_mCD_Prod_v20260220.mesh"
tide_dfs0        = r"examples/tide/Predicted Water Level (CD)_2024_WestCoast_6min.dfs0"
tide_item        = "Predicted tidal elevation"   # item name inside the .dfs0 file
study_area_shp   = None   # optional polygon shapefile to clip study area, or None

# === OUTPUT FILENAMES (change to save different runs side by side) ===
output_dir           = r"output/"
wave_height_map_name = "WaveHeightMap.png"
wave_period_map_name = "WavePeriodMap.png"
wave_params_name     = "wave_params.csv"
shore_impact_name    = "shore_impact.csv"
save_stage_csv       = True   # save {ais_stem}_filtered/depth/wave/impact.csv as checkpoints

# === AIS FILTER PARAMETERS ===
traj_gap_s           = 180.0   # split trajectory on gap > this (seconds)
max_velocity_knots   = 12.0    # kinematic check: flag segment if avg speed exceeds this
max_acceleration_ms2 = 0.2     # acceleration check: replace SOG/COG if implied accel exceeds this
interp_interval_s    = 30.0    # Hermite spline interpolation time step (seconds)

# === WAVE CALCULATION PARAMETERS ===
cb_method      = "L_Le"   # block coefficient method: 'L_Le' (Kriebel default), 'B_Le', or 'table'
                          #   L_Le: Cb estimated from waterline length / bow entry length ratio
                          #   B_Le: uses beam instead of length; table: lookup by vessel type
gravity        = 9.78     # local gravitational acceleration (m/s²); Singapore value (vs 9.81 standard)
max_prop_m     = 2000.0   # maximum wake ray propagation distance from vessel track (m);
                          #   rays beyond this are discarded before coastline intersection
wake_cutoff_m  = 0.01     # minimum shore wave height to record (m); events below this are dropped

# === WAVE FILTER PARAMETERS (which vessel movements produce valid wake events) ===
min_froude_m   = 0.1    # minimum modified Froude number Fr*  (vessel must be in wave-making regime)
max_froude_m   = 0.5    # maximum modified Froude number Fr*  (avoids supercritical planing regime)
max_bf         = 0.4    # maximum BF = β × (Fr*−0.1)²         (no data in Kriebel's dataset exceeds 0.4)
max_sog_knots  = 12.0   # maximum vessel speed (knots)        (fast craft outside calculation range)
max_bl_ratio   = 0.3    # maximum beam / length ratio         (atypical hull form, model not valid)

In [ ]:
# === Cell 2: Build config ===
from aiswakepy.config import load_config

config = load_config({
    "ais": {
        "raw_csv": ais_csv,
        "traj_gap_s": traj_gap_s,
        "max_velocity_knots": max_velocity_knots,
        "max_acceleration_ms2": max_acceleration_ms2,
        "interp_interval_s": interp_interval_s,
        "study_area_shp": study_area_shp,
    },
    "vessel": {"cb_method": cb_method},
    "bathymetry": {
        "source": bathy_file,
        "tide_dfs0": tide_dfs0,
        "tide_item": tide_item,
    },
    "coastline": {"shapefile": coastline_shp},
    "wave": {
        "gravity": gravity,
        "min_froude_m": min_froude_m,
        "max_froude_m": max_froude_m,
        "max_bf": max_bf,
        "max_sog_knots": max_sog_knots,
        "max_bl_ratio": max_bl_ratio,
    },
    "impact": {"max_propagation_m": max_prop_m, "wake_cutoff_m": wake_cutoff_m},
    "output": {
        "directory": output_dir,
        "wave_height_map_name": wave_height_map_name,
        "wave_period_map_name": wave_period_map_name,
        "wave_params_name": wave_params_name,
        "shore_impact_name": shore_impact_name,
        "save_stage_csv": save_stage_csv,
    },
})
print("Config loaded OK")

In [ ]:
# === Cell 3: Stage 1 — AIS Filter ===
from pathlib import Path
from aiswakepy.stages.filter import filter_ais

df_filtered = filter_ais(
    csv_path=config.ais.raw_csv,
    coastline_shp=config.coastline.shapefile,
    gap_s=config.ais.traj_gap_s,
    max_velocity_knots=config.ais.max_velocity_knots,
    max_acceleration_ms2=config.ais.max_acceleration_ms2,
    interval_s=config.ais.interp_interval_s,
    study_area_shp=config.ais.study_area_shp,
)
print(f"Filtered and interpolated: {len(df_filtered):,} rows")
if config.output.save_stage_csv:
    _stem = Path(config.ais.raw_csv).stem
    _out = Path(config.output.directory)
    _out.mkdir(parents=True, exist_ok=True)
    df_filtered.to_csv(_out / f"{_stem}_01_filtered.csv", index=False)
    print(f"  saved {_stem}_01_filtered.csv")
df_filtered.head()

In [ ]:
# === Cell 4: Stage 2 — Depth + Tide Assignment ===
from aiswakepy.stages.depth import assign_depth

df_depth = assign_depth(
    df=df_filtered,
    bathy_path=config.bathymetry.source,
    tide_dfs0_path=config.bathymetry.tide_dfs0,
    tide_item=config.bathymetry.tide_item,
    underkeel_margin_m=config.bathymetry.underkeel_margin_m,
)
print(f"After depth filter: {len(df_depth)} rows")
if config.output.save_stage_csv:
    df_depth.to_csv(_out / f"{_stem}_02_depth.csv", index=False)
    print(f"  saved {_stem}_02_depth.csv")
df_depth[["WaterDepth", "draught"]].describe()

In [ ]:
# === Cell 5: Stage 3 — Wave Parameters ===
from aiswakepy.stages.wave_params import compute_wave_params

print(
    f"Wave calculation filters applied:\n"
    f"  modified Froude Fm : [{config.wave.min_froude_m}, {config.wave.max_froude_m}]\n"
    f"  BF = β(Fm−0.1)²    ≤ {config.wave.max_bf}\n"
    f"  SOG                ≤ {config.wave.max_sog_knots} kn\n"
    f"  beam / length      ≤ {config.wave.max_bl_ratio}"
)

df_wave = compute_wave_params(
    df=df_depth,
    cb_method=config.vessel.cb_method,
    g=config.wave.gravity,
    rho=config.wave.rho_water,
    min_froude_m=config.wave.min_froude_m,
    max_froude_m=config.wave.max_froude_m,
    max_bf=config.wave.max_bf,
    max_sog_knots=config.wave.max_sog_knots,
    max_bl_ratio=config.wave.max_bl_ratio,
)
print(f"Wave events: {len(df_wave)}")
if config.output.save_stage_csv:
    df_wave.to_csv(_out / f"{_stem}_03_wave.csv", index=False)
    print(f"  saved {_stem}_03_wave.csv")
df_wave[["block_coeff", "bow_entry_m", "SOGms", "LengthWL", "Alpha", "Beta", "FroudeM", "FroudeD", "BF", "GHV2", "H_Kriebel", "T", "Emax", "Etot", "Theta", "Cel", "Tc", "BLratio"]].describe()

In [ ]:
# === Cell 6: Stage 4 — Shore Impact ===
from aiswakepy.stages.shore_impact import compute_shore_impact

df_impact = compute_shore_impact(
    df_wave=df_wave,
    coastline_shp=config.coastline.shapefile,
    max_propagation_m=config.impact.max_propagation_m,
    wake_cutoff_m=config.impact.wake_cutoff_m,
    g=config.wave.gravity,
)
print(f"Shore impact events: {len(df_impact)}")
if config.output.save_stage_csv:
    df_impact.to_csv(_out / f"{_stem}_04_impact.csv", index=False)
    print(f"  saved {_stem}_04_impact.csv")
df_impact.head()

In [ ]:
# === Cell 7: Visualisation ===
from pathlib import Path
from aiswakepy.viz.wave_map import plot_wave_height_map, plot_wave_period_map

out = Path(output_dir)
out.mkdir(parents=True, exist_ok=True)

plot_wave_height_map(df_impact, coastline_shp, out / wave_height_map_name, max_points=100_000)
plot_wave_period_map(df_impact, coastline_shp, out / wave_period_map_name, max_points=100_000)
print("Maps saved")

In [ ]:
# === Cell 8: Export final results ===
# shore_impact.csv and stage CSVs are already saved above.
# This cell saves the final wave_params CSV using the configured filename.
from pathlib import Path

out = Path(output_dir)
df_wave.to_csv(out / wave_params_name, index=False)
df_impact.to_csv(out / shore_impact_name, index=False)
print(f"Results saved to {out.resolve()}")